<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/week10_extract_boei_hakusho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 10 · 01 抽取防衛白書＋固定長度 chunking baseline

- 資料：令和8年版 防衛白書 全編（900 頁）https://www.mod.go.jp/j/press/wp/wp2026/pdf/R08zenpen.pdf
- 換資料的原因：SQuAD 答案太短，切法測不出差異；JDocQA 只附單頁文字，PDF 也下載不到（403）
- 評分方式：檢索到的 chunk 只要涵蓋「正解頁」之一，就算命中
- ⚠️ PDF 和 documents.jsonl 不要 commit

**執行順序：從上到下跑一次。之後改題目只要重跑「題目」和「評估」兩格；改切法才需要重跑 chunking 和 embedding。**

In [39]:
!pip install -q pymupdf sentence-transformers

In [40]:
# 確認 GPU 有啟用（要印出 True）。False 的話：執行階段 → 變更執行階段類型 → T4 GPU
import torch
print(torch.cuda.is_available())

True


## 1. 下載 PDF，逐頁抽文字

In [41]:
!wget -q -O /content/hakusho.pdf "https://www.mod.go.jp/j/press/wp/wp2026/pdf/R08zenpen.pdf"
!ls -lh /content/hakusho.pdf

-rw-r--r-- 1 root root 70M Sep  3 02:20 /content/hakusho.pdf


In [42]:
import fitz, json

pdf_path = "/content/hakusho.pdf"
out_path = "/content/documents.jsonl"

doc = fitz.open(pdf_path)
out = open(out_path, "w", encoding="utf-8")

page_number = 1
for page in doc:
  page_text = page.get_text()

  record = {}
  record["doc_id"]="boei_hakusho_2026"
  record["page"] = page_number
  record["text"] = page_text

  line = json.dumps(record, ensure_ascii=False)
  out.write(line + "\n")
  page_number += 1

out.close()
print("Total",len(doc), "pages")


Total 900 pages


In [43]:
pages = []
f = open(out_path, encoding="utf-8")
for line in f:
  one_page = json.loads(line)
  pages.append(one_page)
f.close()




In [44]:
pages = [json.loads(l) for l in open(out_path, encoding="utf-8")]
short = [p for p in pages if len(p["text"].strip()) < 100]
print(f"總頁數 {len(pages)}，幾乎沒有文字的頁數 {len(short)}")
print(pages[499]["text"][:300])   # 第 500 頁（list 從 0 開始）

總頁數 900，幾乎沒有文字的頁數 19
株式会社吉光工業　代表取締役社長　坂
さか
本
もと
　一
かず
樹
き
ものづくりの現場は、製造業経験者や工業高校・高
専といった専門教育を受けた人材中心というイメージ
が強い分野です。しかし、その中にあって愛澤さんのよ
うに異業種から果敢に飛び込み、丁寧な作業と高い品
質意識を強みに、経験年数にとらわれず技術力で評価
を得ている姿は、当社にとって大きな励みであり、もの
づくりの可能性を広げるものです。
当社のような中小企業にとっては、人材の確保と育
成は事業継続に直結する大きな課題です。こうした状
況の中で、企業がどのように人を迎え入れ、成長を促
し、従業員満足度を向上させるかがますます重


## 2. 題目＋正解頁

正解頁 = 所有包含答案關鍵字的 PDF 頁（正文和資料編都算）。
關鍵字比對前先拿掉換行和空格，因為白書的文字會在詞中間換行。

In [45]:
def clean(text):
    text = text.replace("\n", "")      # 刪掉換行
    text = text.replace(" ", "")       # 刪掉空格
    return text


def find(word):
    result = []
    for p in pages:
        if word in clean(p["text"]):
            result.append(p["page"])
    return result


qs = [
    # ---- 原本的 4 題 ----
    {"qid": "q01", "type": "single",
     "question": "防衛事業から撤退する企業が出てくる背景として、白書はどんな事情を挙げている？",
     "keys": ["忌避観", "忌避感"]},
    {"qid": "q02", "type": "list",
     "question": "防衛研究所の研究者が指摘する、防衛産業の構造的なリスク三つは？",
     "keys": ["熟練人材の減少"]},
    {"qid": "q03", "type": "single",
     "question": "機密保持の理由で、海外に頼らず国産を目指すべきとされている分野は？",
     "keys": ["機密保持の観点から外国に依存"]},
    {"qid": "q04", "type": "list",
     "question": "白書が描く防衛産業の「望ましい姿」の三つの柱は？",
     "keys": ["適時かつ十分な量", "防衛装備品の移転を通じて", "デュアルユース分野への積極的な投資"]},

    # ---- 列表型 ----
    {"qid": "q05", "type": "list",
     "question": "装備品をどう取得するか決める際に、白書が前提として挙げている三つの考え方は？",
     "keys": ["比較優位がある分野を育成"]},
    {"qid": "q06", "type": "list",
     "question": "国内に防衛産業を持ち続けることの意義として、従来から言われてきた点は？",
     "keys": ["安全保障上の主体性の確保"]},
    {"qid": "q07", "type": "list",
     "question": "部品の調達網が途切れるリスクに備えて、研究者はどんな対策が必要だと述べている？",
     "keys": ["トレーサビリティの強化"]},
    {"qid": "q08", "type": "list",
     "question": "下請け企業経由の情報流出を防ぐために、供給網全体で求められている取り組みは？",
     "keys": ["供給網全体を対象としたセキュリティ基準"]},
    {"qid": "q09", "type": "list",
     "question": "他国への装備品の提供は、どのような目的を果たすための手段と位置づけられている？",
     "keys": ["国際法に違反する侵略や武力の行使"]},

    # ---- 事實／數字 ----
    {"qid": "q10", "type": "single",
     "question": "艦艇一隻の製造にかかわる企業の数は、どれくらいとされている？",
     "keys": ["8,000社以上"]},
    {"qid": "q11", "type": "single",
     "question": "2023年に公表された基盤強化の方針は、それ以前のどの文書に取って代わった？",
     "keys": ["防衛生産・技術基盤戦略」に代わり"]},
    {"qid": "q12", "type": "single",
     "question": "2021年に明らかになった大規模なサイバー攻撃では、国内のどれくらいの組織が狙われた？",
     "keys": ["約200の企業"]},
    {"qid": "q13", "type": "single",
     "question": "防衛装備庁が、新しく防衛分野に入りたい企業向けの展示会を始めたのはいつから？",
     "keys": ["マッチングを図る展示会"]},
    {"qid": "q14", "type": "single",
     "question": "無人機を使って実際の機雷を処分する試験が初めて行われたのは、いつどこで？",
     "keys": ["無人機による実機雷処分"]},

    # ---- 白書に答えがない ----
    {"qid": "q15", "type": "none",
     "question": "防衛装備庁の職員の平均年収はいくら？",
     "keys": []},
]

for q in qs:
    found_pages = []
    for word in q["keys"]:
        for pg in find(word):
            if pg not in found_pages:     # 不要重複
                found_pages.append(pg)
    found_pages.sort()
    q["pages"] = found_pages
    print(q["qid"], q["type"], q["pages"])

    # 有答案的題目卻找不到正解頁 → 停下來
    if len(found_pages) == 0 and q["type"] != "none":
        raise ValueError(q["qid"] + " 找不到正解頁，換一個關鍵字")

q01 single [490, 820, 828]
q02 list [491]
q03 single [493, 821]
q04 list [493]
q05 list [492, 821]
q06 list [492]
q07 list [491]
q08 list [491]
q09 list [487, 493, 520, 522, 618, 638, 822, 825, 837, 839, 841, 844]
q10 single [490, 518]
q11 single [492, 819]
q12 single [490, 820]
q13 single [501]
q14 single [292, 368]
q15 none []


## 3. 固定長度 chunking（baseline，不清洗文字）

In [46]:
# 把 900 頁接成一整串文字，同時記住每個字來自第幾頁
full = ""
char_page = []
for p in pages:
    full += p["text"]
    char_page += [p["page"]] * len(p["text"])

def chunk_fixed(size, overlap):
    chunks = []
    step = size - overlap
    for start in range(0, len(full), step):
        end = min(start + size, len(full))
        chunks.append({"text": full[start:end],
                       "pages": sorted(set(char_page[start:end]))})  # 這塊涵蓋哪幾頁
        if end == len(full):
            break
    return chunks

chunks = chunk_fixed(400, 60)
print(len(chunks), "塊")

4545 塊


## 4. Embedding（慢：GPU 約 4 分鐘。只有改切法時才重跑）

In [47]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")

emb = model.encode([c["text"] for c in chunks], normalize_embeddings=True,
                   batch_size=32, show_progress_bar=True)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/143 [00:00<?, ?it/s]

## 5. 評估：top-k 是否命中正解頁

In [48]:
def top_k_chunks(question, k):
    q_vec = model.encode([question], normalize_embeddings=True)[0]

    score_list = []
    for j in range(len(chunks)):
        score = float(q_vec @ emb[j])      # 問題和第 j 塊的相似度
        score_list.append((score, j))

    score_list.sort(reverse=True)          # 分數高的排前面
    top = []
    for score, j in score_list[:k]:
        top.append(j)
    return top, score_list


def chunk_has_page(j, q):
    for pg in chunks[j]["pages"]:
        if pg in q["pages"]:
            return True
    return False


def chunk_has_answer(j, q):
    text = clean(chunks[j]["text"])
    for word in q["keys"]:
        if word in text:
            return True
    return False


k = 5
page_hits = 0
answer_hits = 0
answerable = 0

for q in qs:
    if q["type"] == "none":
        print(q["qid"], "（沒有答案的題目，留到第 12 週測拒答）")
        continue

    answerable = answerable + 1
    top, score_list = top_k_chunks(q["question"], k)

    page_hit = False
    answer_hit = False
    for j in top:
        if chunk_has_page(j, q):
            page_hit = True
        if chunk_has_answer(j, q):
            answer_hit = True

    if page_hit:
        page_hits = page_hits + 1
    if answer_hit:
        answer_hits = answer_hits + 1

    # 正確的那一塊排第幾名（用頁碼判斷）
    rank = None
    position = 1
    for score, j in score_list:
        if chunk_has_page(j, q):
            rank = position
            break
        position = position + 1

    print(q["qid"], q["type"], "正解頁", q["pages"],
          "| 頁命中", page_hit, "| 答案命中", answer_hit, "| 排第", rank, "名")

print()
print("頁命中 recall@" + str(k), "=", page_hits, "/", answerable)
print("答案命中 recall@" + str(k), "=", answer_hits, "/", answerable)

q01 single 正解頁 [490, 820, 828] | 頁命中 True | 答案命中 True | 排第 1 名
q02 list 正解頁 [491] | 頁命中 True | 答案命中 True | 排第 1 名
q03 single 正解頁 [493, 821] | 頁命中 True | 答案命中 True | 排第 1 名
q04 list 正解頁 [493] | 頁命中 False | 答案命中 False | 排第 15 名
q05 list 正解頁 [492, 821] | 頁命中 True | 答案命中 True | 排第 1 名
q06 list 正解頁 [492] | 頁命中 False | 答案命中 False | 排第 12 名
q07 list 正解頁 [491] | 頁命中 True | 答案命中 True | 排第 5 名
q08 list 正解頁 [491] | 頁命中 True | 答案命中 True | 排第 1 名
q09 list 正解頁 [487, 493, 520, 522, 618, 638, 822, 825, 837, 839, 841, 844] | 頁命中 True | 答案命中 True | 排第 1 名
q10 single 正解頁 [490, 518] | 頁命中 True | 答案命中 True | 排第 1 名
q11 single 正解頁 [492, 819] | 頁命中 True | 答案命中 False | 排第 4 名
q12 single 正解頁 [490, 820] | 頁命中 True | 答案命中 True | 排第 1 名
q13 single 正解頁 [501] | 頁命中 True | 答案命中 True | 排第 1 名
q14 single 正解頁 [292, 368] | 頁命中 True | 答案命中 True | 排第 3 名
q15 （沒有答案的題目，留到第 12 週測拒答）

頁命中 recall@5 = 12 / 14
答案命中 recall@5 = 11 / 14


In [49]:
for pg in [820, 830]:
    t = pages[pg-1]["text"].replace("\n", "").replace(" ", "")
    print(pg, "望ましい姿" in t, "忌避" in t)
    print(t[:300], "\n")

820 False True
子技術に注目が集まっている。特に、量子暗号通信や量子センサ、量子コンピュータ、更に量子コンピュータでは解読できない耐量子計算機暗号といった、軍事分野への応用が期待されている技術の実用化に向けた研究開発が各国において進められている。また、3Dプリンタのような積層製造技術も実用化が加速している。積層製造技術を活用することにより、複雑な構造物の製造が低コストで可能となり、在庫に頼らない部品調達によって兵站に革命が起きる可能性があるとされている。近年開発が進められている極超音速滑空兵器は、通常の弾道ミサイルとは異なる低い軌道を、マッハ5を超える極超音速で長時間飛翔すること、機動性を有すること等から、探 

830 False False
資料73防衛技術指針2023（ExecutiveSummary）策定の趣旨防衛技術指針2023は、「国家安全保障戦略」、「国家防衛戦略」及び「防衛力整備計画」の策定を受け、「防衛技術基盤の強化」の方針を具体化するものであり、各種の取組を省として一体的かつ強力に推進する際の指針となるものである。また、防衛省における防衛技術基盤の強化の方針を省外にも発信することで、企業等の予見可能性を高めるとともに、防衛技術基盤の強化についての共通認識を醸成し、技術的な連携を強力に進める基盤の構築を目指す。防衛技術基盤の現状と課題科学技術の進展は、我が国に経済的・社会的発展をもたらすとともに、安全保障環境にも大き 



In [50]:
t = pages[819]["text"].replace("\n", "").replace(" ", "")
i = t.find("忌避")
print(t[i-150:i+100])

が必要となる。また、これら企業は得てして独自仕様、少量多種生産を求められ、装備品等の運用期間の長さから、長期にわたる製造等の体制確保も必要となる。さらに、顧客は基本的には防衛省・自衛隊に限定されることもあり、企業にとって投資回収の機会は限られる。加えて、依然として日本社会においては、防衛事業に対する忌避感やレピュテーションリスクといった問題がある。上記のような特徴にも起因し、基盤の弱体化が進んでいる。高度な要求性能や保全措置への対応の必要性等により、多大な経営資源の投入を必要とする一方、収益性は


In [51]:
def rank_of_correct(q):
    s = (model.encode([q["question"]], normalize_embeddings=True) @ emb.T)[0]
    order = s.argsort()[::-1]
    for r, j in enumerate(order, start=1):
        if set(chunks[j]["pages"]) & set(q["pages"]):
            return r, chunks[j]["text"][:200]

for q in qs:
    print(q["qid"], rank_of_correct(q))

q01 (1, 'そ\nうした中、防衛事業からの撤退や事業規模の縮小を決断す\nる企業が断続的に現れており、加えて、既存の企業による\n新たな投資や新規参入も低調になりがちである。その結\n果、自衛隊の運用に必要不可欠な装備品等の安定的な調達\nに支障が生じるだけでなく、長期的には、適正な競争環境\nやイノベーションが失われ、安全保障分野における我が国\nの技術的優位性を喪失するおそれもある。\nさらに、近年、基盤を取り巻く様々な')
q02 (1, 'なりました。防衛関連企業の場合、設計図面や試験\nデータが流出すれば、技術的優位性の喪失のみならず、\n国家安全保障上の重大な損失となります。供給網全体\nを対象としたセキュリティ基準の統一、監査制度の導\n入、ゼロトラスト型の情報管理体制の構築が求められ\nますが、中小企業にとってはコスト負担が重く、実装面\nでの支援策が不可欠です。\nこれら3つの課題は相互に連関しています。今後は、\n効率性と冗長性の再均衡')
q03 (1, 'と平素からの運用、維持\n整備に係る改善能力の確保の観点から不可欠なもの\n（例：弾薬、艦船）\nウ\x07\u3000機密保持の観点から外国に依存すべきでないもの\n（例：通信、暗号技術）\nエ\x07\u3000我が国の地理的、政策的な特殊性を踏まえた運用構\n想の実現に不可欠なもの\nオ\u3000外国からの最新技術の入手が困難なもの\nカ\x07\u3000経済的手段による外的脅威の対象となり得るもの\nまた、国産による取得により難い場合であっても、我が\n国への技')
q04 (15, 'う事業者。\n主な販路が防衛省に限られることや利益率の低さなどか\nら、成長が見込めないと考えられてきた。しかし、従来\nの枠組みにとどまることなく、能動的な成長投資を行\nい、装備品の移転やデュアルユースによる民生市場の獲\n得などに積極的に取り組んでいくことが重要である。\nこのような認識のもと、次に示す防衛産業の望ましい\n姿の実現に向けて、官民が連携して取り組んでいく。\n①\u3000防衛力の強化に必要な装備品等')
q05 (1, '点から、基盤の維持・強化のための方策\nを講じるに当たっては、完成品としての装備品等のみなら\nず、それに用いる部品・構成品の安定的な製造等の確保も

In [52]:
import re

def is_heading(line):
    # 判斷這一行是不是「大標題」（可以在這裡切）
    line = line.strip()
    if line == "":
        return False
    # （1）（2）（3）…
    if re.match(r"^（\d+）", line):
        return True
    # 第1節、第 2 章…
    if re.match(r"^第\s*\d+\s*[章節]", line):
        return True
    # 注意：①②③ 和 アイウ 是列表項目，不當切點
    return False


def chunk_structure(max_size, min_size):
    chunks = []
    current_text = ""
    current_pages = []

    for p in pages:
        for line in p["text"].split("\n"):

            # 規則 1：遇到大標題，而且目前這塊已經夠長 → 先收起來
            if is_heading(line) and len(current_text) >= min_size:
                chunks.append({"text": current_text, "pages": current_pages})
                current_text = ""
                current_pages = []

            # 規則 2：再加這一行就太長 → 先收起來
            if len(current_text) + len(line) > max_size and current_text != "":
                chunks.append({"text": current_text, "pages": current_pages})
                current_text = ""
                current_pages = []

            # 把這一行加進目前這塊
            current_text = current_text + line + "\n"
            if p["page"] not in current_pages:
                current_pages.append(p["page"])

    # 最後剩下的也要收
    if current_text != "":
        chunks.append({"text": current_text, "pages": current_pages})
    return chunks


test = chunk_structure(1000, 200)
print("切成", len(test), "塊")

切成 2005 塊


In [53]:
def has_page(chunk, q):
    for pg in chunk["pages"]:
        if pg in q["pages"]:
            return True
    return False


def has_answer(chunk, q):
    text = clean(chunk["text"])
    for word in q["keys"]:
        if word in text:
            return True
    return False


def evaluate(chunks, emb, k):
    page_hits = 0
    answer_hits = 0
    answerable = 0
    ranks = {}

    for q in qs:
        if q["type"] == "none":
            continue
        answerable = answerable + 1

        q_vec = model.encode([q["question"]], normalize_embeddings=True)[0]
        score_list = []
        for j in range(len(chunks)):
            score_list.append((float(q_vec @ emb[j]), j))
        score_list.sort(reverse=True)

        page_hit = False
        answer_hit = False
        for score, j in score_list[:k]:
            if has_page(chunks[j], q):
                page_hit = True
            if has_answer(chunks[j], q):
                answer_hit = True
        if page_hit:
            page_hits = page_hits + 1
        if answer_hit:
            answer_hits = answer_hits + 1

        rank = None
        position = 1
        for score, j in score_list:
            if has_page(chunks[j], q):
                rank = position
                break
            position = position + 1
        ranks[q["qid"]] = rank

    return page_hits, answer_hits, answerable, ranks

In [54]:
configs = [
    {"name": "fixed_400_60",   "method": "fixed",     "size": 400,  "overlap": 60},
    {"name": "fixed_800_120",  "method": "fixed",     "size": 800,  "overlap": 120},
    {"name": "fixed_1000_150", "method": "fixed",     "size": 1000, "overlap": 150},
    {"name": "structure_1000", "method": "structure", "size": 1000, "min": 200},
]

results = []
for cfg in configs:
    print("=== 開始", cfg["name"], "===")

    # 1. 切
    if cfg["method"] == "fixed":
        cs = chunk_fixed(cfg["size"], cfg["overlap"])
    else:
        cs = chunk_structure(cfg["size"], cfg["min"])

    # 2. 算平均長度
    texts = []
    total_len = 0
    for c in cs:
        texts.append(c["text"])
        total_len = total_len + len(c["text"])
    avg_len = total_len / len(cs)

    # 3. embedding
    e = model.encode(texts, normalize_embeddings=True,
                     batch_size=32, show_progress_bar=True)

    # 4. 評估
    ph, ah, n, ranks = evaluate(cs, e, 5)
    results.append({"name": cfg["name"], "n_chunks": len(cs), "avg_len": avg_len,
                    "page": ph, "answer": ah, "n": n, "ranks": ranks})

# 5. 印出比較表
print()
print("設定 | 塊數 | 平均長度 | 頁命中 | 答案命中 | q04名次 | q06名次")
for r in results:
    print(r["name"], "|", r["n_chunks"], "|", round(r["avg_len"]), "|",
          str(r["page"]) + "/" + str(r["n"]), "|",
          str(r["answer"]) + "/" + str(r["n"]), "|",
          r["ranks"]["q04"], "|", r["ranks"]["q06"])

=== 開始 fixed_400_60 ===


Batches:   0%|          | 0/143 [00:00<?, ?it/s]

=== 開始 fixed_800_120 ===


Batches:   0%|          | 0/72 [00:00<?, ?it/s]

=== 開始 fixed_1000_150 ===


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

=== 開始 structure_1000 ===


Batches:   0%|          | 0/63 [00:00<?, ?it/s]


設定 | 塊數 | 平均長度 | 頁命中 | 答案命中 | q04名次 | q06名次
fixed_400_60 | 4545 | 400 | 12/14 | 11/14 | 15 | 12
fixed_800_120 | 2273 | 800 | 11/14 | 11/14 | 10 | 19
fixed_1000_150 | 1818 | 1000 | 12/14 | 11/14 | 3 | 6
structure_1000 | 2005 | 771 | 12/14 | 11/14 | 5 | 1


In [55]:
qids = []
for q in qs:
    if q["type"] != "none":
        qids.append(q["qid"])

header = "題目"
for r in results:
    header = header + " | " + r["name"]
print(header)

for qid in qids:
    row = qid
    for r in results:
        row = row + " | " + str(r["ranks"][qid])
    print(row)

題目 | fixed_400_60 | fixed_800_120 | fixed_1000_150 | structure_1000
q01 | 1 | 2 | 4 | 2
q02 | 1 | 1 | 1 | 1
q03 | 1 | 1 | 1 | 1
q04 | 15 | 10 | 3 | 5
q05 | 1 | 1 | 1 | 1
q06 | 12 | 19 | 6 | 1
q07 | 5 | 2 | 3 | 4
q08 | 1 | 1 | 3 | 11
q09 | 1 | 1 | 1 | 1
q10 | 1 | 1 | 1 | 2
q11 | 4 | 22 | 5 | 4
q12 | 1 | 1 | 9 | 1
q13 | 1 | 1 | 1 | 1
q14 | 3 | 3 | 2 | 13


In [56]:
for qid in ["q08", "q14"]:
    for q in qs:
        if q["qid"] == qid:
            target = q
    for c in cs:
        if has_answer(c, target):
            print(qid, "長度", len(c["text"]), "頁", c["pages"])
            print(c["text"][:400])
            print("-----")

q08 長度 335 頁 [491]
国家安全保障上の重大な損失となります。供給網全体
を対象としたセキュリティ基準の統一、監査制度の導
入、ゼロトラスト型の情報管理体制の構築が求められ
ますが、中小企業にとってはコスト負担が重く、実装面
での支援策が不可欠です。
これら3つの課題は相互に連関しています。今後は、
効率性と冗長性の再均衡を図りつつ、防衛産業を国家
安全保障の基盤インフラとして再定義する視座が求め
られます。経済合理性だけでは測れない「安全保障とし
ての産業政策」が、今まさに問われています。
視点
わが国の防衛産業が直面する課題
（注）本コラムは、研究者個人の立場から学術的な分析を述べたものであり、その内容は政府としての公式見解を示すものではありません。
481
令和8年版 防衛白書

-----
q14 長度 995 頁 [292, 293]
艦艇から発射。対水上戦において敵艦
艇等を攻撃。
⑦艦艇攻撃用UAV（空自）
長距離を飛行しつつ、敵艦艇等を
探知・識別し、攻撃。
海自UAV
海自UAV
空自UAV
空自UAV
陸自UAV
陸自UAV
USV・UUV
USV・UUV
⑧レーダーサイト防衛用UAV（空自）
レーダーサイトに飛来するUAV等を探知・識別し、攻撃。
⑤
⑥
⑦
⑧
①
③
④
②
⑨
⑩
⑩
UAV : Unmanned Aerial Vehicle   USV : Unmanned Surface Vehicle   UUV : Unmanned Underwater Vehicle
多層的沿岸防衛体制（SHIELD）（イメージ）
国全体の防衛体制の強化 
▶P.341
⿟政府は、防衛力の抜本的強化に加えて、外交力、情報力、経済力、技術力を含めた国力を統合し、あらゆる政策手段を体
系的に組み合わせて国全体の防
-----
q14 長度 513 頁 [367, 368]
第4節
わが国自身による防衛体制を強化するための訓練・演習など

内にはない良好な演習基盤を活用した実動訓練への参加
を通じて、戦術技量の向上を図っている。
また、師団レベルや方面隊レベルの実動演習では、限
られた国内の演習場などを最大限に活用しているほか、
地元の理解と協力を獲得しながら自衛隊施設・区域以外
を活用した、より実戦的な訓練を行っている。
（2）海上自

## 筆記

- 第一次跑全部 0%：原因是題目被重新定義、正解頁變成空的（評分程式的 bug，不是檢索失敗）。加了 assert 防止再發生
- 抽取雜訊：振假名被拆開、頁首頁尾混入、雙欄排版造成閱讀順序錯亂、換行不等於段落
- 4 題只能確認流程跑得通，要比較切法至少需要 15–30 題